In [ ]:
import streamlit as st
import pandas as pd
from openai import AzureOpenAI

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://jobs-model.openai.azure.com/",
    api_key='',
)

deployment_name = "gpt-4o-mini"

# --- Agent class ---
class Agent:
    def __init__(self, name, system_prompt, client, deployment_name):
        self.name = name
        self.system_prompt = system_prompt
        self.client = client
        self.deployment_name = deployment_name

    def run(self, user_input):
        response = self.client.chat.completions.create(
            model=self.deployment_name,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_input}
            ],
            max_tokens=1000
        )
        return response.choices[0].message.content

# --- Prompts ---
job_loader_prompt = """
You are a data loader agent.
Your job is to:
- Load dataset
- Clean missing values
- Return summary of dataset
"""
skills_prompt = """
You are a skills analyst.
Identify:
- Top 10 most in-demand skills
- Frequency of each skill
- Skill clusters (AI, Data, Cloud, etc.)
"""
trend_prompt = """
You are a trend forecaster.
Based on the skills:
- Identify emerging skills
- Predict future demand
- Highlight declining skills
"""
report_prompt = """
You are a report writer.
Generate a structured report with:
- Key insights
- Top skills
- Career recommendations
- Summary
"""

# --- Streamlit UI ---
st.title("Job Skills Trend Analyzer")

uploaded_file = st.file_uploader("Upload CSV of job postings", type="csv")

if uploaded_file:
    df = pd.read_csv(uploaded_file)
    st.write("Dataset Preview", df.head())

    if st.button("Run Multi-Agent Analysis"):
        # Initialize agents
        job_loader = Agent("Job Loader", job_loader_prompt, client, deployment_name)
        skills_agent = Agent("Skills Analyst", skills_prompt, client, deployment_name)
        trend_agent = Agent("Trend Forecaster", trend_prompt, client, deployment_name)
        report_agent = Agent("Report Writer", report_prompt, client, deployment_name)

        with st.spinner("Running analysis..."):
            try:
                data_summary = job_loader.run(df.head(50).to_string())
                skills_output = skills_agent.run(data_summary)
                trend_output = trend_agent.run(skills_output)
                final_report = report_agent.run(trend_output)

                st.subheader("Final Report")
                st.text(final_report)

            except Exception as e:
                st.error(f"Error: {e}")

### AI-Powered Job Market Prediction Model Using Azure Generative AI